In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
!jupyter labextension list

JupyterLab v4.6.2
C:\Users\user\Documents\agentic-tutor-rag\agentic-tutor-rag\.venv\share\jupyter\labextensions
        jupyterlab-plotly v6.9.0 enabled ok (python, plotly)
        jupyterlab_pygments v0.3.0 enabled ok (python, jupyterlab_pygments)
        @jupyter-notebook/lab-extension v7.6.1 enabled ok
        @jupyter-widgets/jupyterlab-manager v5.0.15 enabled ok (python, jupyterlab_widgets)



In [2]:
!jupyter lab --version

4.6.2


In [2]:
import sys
import os

# Ensure Python can find your 'modules' folder
sys.path.append(os.path.abspath('..'))

from modules.ground_truth import generate_ground_truth_dataset

# Define where your files live
input_path = 'data/knowledge-base.json' # Change this if your file is somewhere else!
output_path = 'data/ground_truth.json'

# Let it run! (This might take a minute depending on how big your JSON is)
generate_ground_truth_dataset(input_filepath=input_path, output_filepath=output_path)

Generating questions for 116 documents...


  0%|          | 0/116 [00:00<?, ?it/s]


Done! Saved 116 records to data/ground_truth.json


In [3]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# Import your custom modules
import sys
sys.path.append(os.path.abspath('..')) # Ensures it can find 'modules' if running in notebooks/
from modules.ingest import load_faq_data, build_indices
from modules.rag_helper import RAGBase, VectorRAG

# 1. Load Data & Build Both Retrievers
# Note: Adjust the path inside load_faq_data() if it can't find knowledge-base.json
documents = load_faq_data() 
keyword_index, vector_index = build_indices(documents)

client = OpenAI()
keyword_rag = RAGBase(index=keyword_index, llm_client=client)
vector_rag = VectorRAG(index=vector_index, llm_client=client)

# 2. Load Your Newly Generated Ground Truth Dataset
# This path check ensures it works whether you run it from the root or the notebooks folder
gt_path = '../data/ground_truth.json' if os.path.exists('../data/ground_truth.json') else 'data/ground_truth.json'

with open(gt_path, 'r') as f:
    ground_truth = json.load(f)

print(f"Loaded {len(ground_truth)} test questions from ground truth.")

# 3. Dedicated MRR & Hit Rate Evaluation Function
def calculate_mrr_and_hit_rate(rag_system, dataset, num_results=5):
    total_queries = len(dataset)
    hits = 0
    reciprocal_ranks = []

    for item in tqdm(dataset, desc="Evaluating Retriever"):
        question = item['user_query']
        expected_chunk = item['chunk_id']
        
        # Perform retrieval using just the search method
        results = rag_system.search(question, num_results=num_results)
        
        # Find position of expected chunk
        found_rank = 0
        for rank, doc in enumerate(results, start=1):
            if doc.get('chunk_id') == expected_chunk:
                found_rank = rank
                break
        
        # Calculate Hits and MRR math
        if found_rank > 0:
            hits += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

    hit_rate = hits / total_queries
    mrr = sum(reciprocal_ranks) / total_queries
    
    return hit_rate, mrr

# 4. Run Evaluation on Keyword Search
print("\n--- Evaluating Keyword Search (minsearch) ---")
kw_hit_rate, kw_mrr = calculate_mrr_and_hit_rate(keyword_rag, ground_truth)

# 5. Run Evaluation on Vector Search
print("\n--- Evaluating Vector Search (FastEmbed) ---")
vec_hit_rate, vec_mrr = calculate_mrr_and_hit_rate(vector_rag, ground_truth)

# 6. Summary Comparison Table
results_df = pd.DataFrame([
    {
        "Retriever": "Keyword Search (minsearch)", 
        "Hit Rate (@5)": f"{kw_hit_rate * 100:.2f}%", 
        "MRR": f"{kw_mrr:.4f}"
    },
    {
        "Retriever": "Vector Search (FastEmbed)", 
        "Hit Rate (@5)": f"{vec_hit_rate * 100:.2f}%", 
        "MRR": f"{vec_mrr:.4f}"
    }
])

print("\n=== RETRIEVAL EVALUATION SUMMARY ===")
print(results_df.to_string(index=False))

Building Keyword Index...
Building Vector Index...
Both indices built successfully!
Loaded 116 test questions from ground truth.

--- Evaluating Keyword Search (minsearch) ---


Evaluating Retriever:   0%|          | 0/116 [00:00<?, ?it/s]


--- Evaluating Vector Search (FastEmbed) ---


Evaluating Retriever:   0%|          | 0/116 [00:00<?, ?it/s]


=== RETRIEVAL EVALUATION SUMMARY ===
                 Retriever Hit Rate (@5)    MRR
Keyword Search (minsearch)        62.07% 0.3996
 Vector Search (FastEmbed)        77.59% 0.5188



### === RETRIEVAL EVALUATION SUMMARY ===  
                 Retriever   Hit Rate (@5)    MRR  
  Keyword Search (minsearch)--------60.00%     0.3752  
 Vector Search (FastEmbed) ----------      71.30%     0.4277  

In [3]:
import os
import json
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
import sys

# Ensure it can find 'modules'
sys.path.append(os.path.abspath('..')) 
from modules.ingest import load_faq_data, build_indices

# ONLY import AdvancedRAG (You don't need VectorRAG anymore!)
from modules.rag_helper import AdvancedRAG

# 1. Load Data & Build Indices
documents = load_faq_data() 
keyword_index, vector_index = build_indices(documents)

client = OpenAI()

# Initialize AdvancedRAG
advanced_rag = AdvancedRAG(
    vector_index=vector_index, 
    keyword_index=keyword_index, 
    llm_client=client, 
    prompt_template="" 
)

# 2. Load Ground Truth
gt_path = '../data/ground_truth.json' if os.path.exists('../data/ground_truth.json') else 'data/ground_truth.json'
with open(gt_path, 'r') as f:
    ground_truth = json.load(f)

print(f"Loaded {len(ground_truth)} test questions.")

# 3. Dedicated MRR & Hit Rate Evaluation Function
def calculate_mrr_and_hit_rate(search_function, dataset, num_results=5):
    total_queries = len(dataset)
    hits = 0
    reciprocal_ranks = []

    for item in tqdm(dataset, desc="Evaluating Retriever"):
        question = item['user_query']
        expected_chunk = item['chunk_id']
        
        # Call the search function passed into the loop
        results = search_function(question, num_results=num_results)
        
        # Find position of expected chunk
        found_rank = 0
        for rank, doc in enumerate(results, start=1):
            if doc.get('chunk_id') == expected_chunk:
                found_rank = rank
                break
        
        # Calculate Hits and MRR math
        if found_rank > 0:
            hits += 1
            reciprocal_ranks.append(1.0 / found_rank)
        else:
            reciprocal_ranks.append(0.0)

    return (hits / total_queries), (sum(reciprocal_ranks) / total_queries)

# ---------------------------------------------------------
# THE MAGIC HAPPENS HERE: Define the search methods
# ---------------------------------------------------------

# A. The OLD Way (Just Vector Search)
def baseline_vector_search(query, num_results=5):
    # We just use the vector_index directly! No VectorRAG class needed.
    return vector_index.search(query, num_results=num_results)

# B. The NEW Way (Advanced RAG: Rewrite -> Hybrid -> Rerank)
def old_advanced_search(query, num_results=5):
    opt_query = advanced_rag.rewrite_query(query)
    docs = advanced_rag.hybrid_search(opt_query, top_k=10)
    return advanced_rag.rerank_documents(opt_query, docs, top_n=num_results)
    
def agentic_advanced_search(query, num_results=5):
    search_plan = advanced_rag.agentic_query_planner(query)
    docs = advanced_rag.agentic_hybrid_search(search_plan, top_k=10)
    return advanced_rag.rerank_documents(query, docs, top_n=num_results)

# 4. Run the Evaluations!
print("\n--- 1. Evaluating Baseline (Vector Only) ---")
vec_hit, vec_mrr = calculate_mrr_and_hit_rate(baseline_vector_search, ground_truth)

print("\n--- 2. Evaluating Old Advanced RAG ---")
old_adv_hit, old_adv_mrr = calculate_mrr_and_hit_rate(old_advanced_search, ground_truth)

print("\n--- 3. Evaluating NEW Agentic RAG ---")
agent_hit, agent_mrr = calculate_mrr_and_hit_rate(agentic_advanced_search, ground_truth)

# Summary Comparison Table
results_df = pd.DataFrame([
    {"System": "1. Baseline (Vector Only)", "Hit Rate": f"{vec_hit * 100:.2f}%", "MRR": f"{vec_mrr:.4f}"},
    {"System": "2. Old Advanced RAG", "Hit Rate": f"{old_adv_hit * 100:.2f}%", "MRR": f"{old_adv_mrr:.4f}"},
    {"System": "3. New Agentic RAG", "Hit Rate": f"{agent_hit * 100:.2f}%", "MRR": f"{agent_mrr:.4f}"}
])

print("\n=== ULTIMATE RETRIEVAL LEADERBOARD ===")
print(results_df.to_string(index=False))

Building Keyword Index...
Building Vector Index...
Both indices built successfully!
Loaded 116 test questions.

--- 1. Evaluating Baseline (Vector Only) ---


Evaluating Retriever:   0%|          | 0/116 [00:00<?, ?it/s]


--- 2. Evaluating Old Advanced RAG ---


Evaluating Retriever:   0%|          | 0/116 [00:00<?, ?it/s]


--- 3. Evaluating NEW Agentic RAG ---


Evaluating Retriever:   0%|          | 0/116 [00:00<?, ?it/s]


=== ULTIMATE RETRIEVAL LEADERBOARD ===
                   System Hit Rate    MRR
1. Baseline (Vector Only)   77.59% 0.5188
      2. Old Advanced RAG   38.79% 0.2464
       3. New Agentic RAG   68.97% 0.3536
